<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/llm_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [5]:
# @title Dependencies

# Installation
! pip install pandas pyarrow -q
! pip install numpy -q
! pip install openai -q

# first-party installations
import itertools
from typing import Dict, Any, Tuple, Optional

# third-party installations
from google.colab import drive
import pandas as pd
import numpy as np
from openai import OpenAI
from openai import APIError

In [2]:
# @title Paths and definitions

# mount GDrive
drive.mount('/content/drive')

# define dataset directory
DATASET_DIR = "/content/drive/MyDrive/Thesis/dataset/"

# define models (as used by Cummins)

# all models using the OpenAI-API
OPENAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-08-06",
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"
]

# all models using the Groq-API
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "deepseek-r1-distill-llama-70b"
]

# models who can vary by reasoning effort
REASONING_MODELS = {
    "o4-mini-2025-04-16",
    "gpt-5-mini-2025-08-07",
    "o3-mini-2025-01-31",
    "gpt-5-nano-2025-08-07"
}


Mounted at /content/drive


In [3]:
# @title Load data

# define target column
TARGET_COL = "parsed_text"

# read dataset_paper
dataset_paper = pd.read_parquet(DATASET_DIR + "dataset_paper.parquet")

# select columns 'paper_id', 'abstract' and the target column 'parsed_text'
paper_content = dataset_paper[['paper_id', 'abstract', TARGET_COL]]


In [4]:
# @title API settings

# models approached by the OpenAI API
OpenAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-08-06",
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"]

# OpenAI API-key
OPENAI_KEY = "api_key_placeholder"

# models approached by the GROQ API
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "deepseek-r1-distill-llama-70b"]

# GROQ API-key
GROQ_KEY = "api_key_placeholder"

# GROQ URL
GROQ_URL ="https://api.groq.com/openai/v1"

# Prepare content

In [ ]:
# @title Create configurations

# complete list of models
MODELS = OPENAI_MODELS + GROQ_MODELS

# define temperature parameters (as used by Cummins)
TEMPS = [0.0, 0.5, 1.0, 1.5]

# define reasoning parameters (as used by Cummins)
REASONING = ["low", "high"]

# Prompt style (CoT) (as used by Li et al. 2025)
CoT = [
    "",
    "explain your thought process step by step"
]

# generates every single combination
raw_grid = list(itertools.product(MODELS, TEMPS, REASONING, CoT))
grid_df = pd.DataFrame(raw_grid, columns=["model", "temperature", "reasoning_effort", "CoT"])

# conditional deletion weather a model has a temperature or reasoning paramter
grid_df["reasoning_effort"] = np.where(
    ~grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["reasoning_effort"]
)
grid_df["temperature"] = np.where(
    grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["temperature"]
)

# drop duplicates
experiment_config = grid_df.drop_duplicates(ignore_index=True)

print(f"Total valid parameter combos: {len(experiment_config)}")
display(experiment_config)

print("\nConfiguration table completed!")

# API

In [ ]:
# @title Define API-calls
# self-created

try:

    # retrieve keys
    openai_api_key = OPENAI_KEY
    groq_api_key = GROQ_KEY

    # initialize the OpenAI client
    openai_client = OpenAI(api_key=openai_api_key)

    # initialize the GROQ client
    groq_client = OpenAI(
        api_key=groq_api_key,
        groq_url=GROQ_URL
    )

    # message after initialization
    print("API Clients initialized successfully.")

except Exception as e:
    # error message
    print(f"ERROR during client initialization: {e}")
    # raise error
    raise RuntimeError(f"Client initialization failed: {e}")

for config in experiment_config:

    model_name = config['model'] # dimension

    raw_temp = config.get('temperature') # dimension
    temp = raw_temp if pd.notna(raw_temp) else None

    raw_reasoning_effort = config.get('reasoning_effort')
    reasoning_effort = raw_reasoning_effort if pd.notna(raw_reasoning_effort) else None ### The API might not be able to process this argument!

    CoT_level = config['CoT']

    # message of current model
    print(f"\nProcessing model: {model_name}")

    # client selection
    if model_name in GROQ_MODELS:
        client_to_use = groq_client
        print(f"  --> Selected GROQ client.")
    else:
        client_to_use = openai_client
        print(f"  --> Selected OPENAI client.")

    # basic content of the API-call
    completion_kwargs = {
        'model': model_name,
        'messages': [
            {"role": "user", "content": prompt_content} ### What shouuld be the user content? Like in Jamies Colab?
        ],
    }

    # Add temperature if it is defined
    if temp is not None:
        completion_kwargs['temperature'] = temp

    # Conditionally add reasoning_effort (using a hypothetical parameter name)
    if reasoning_effort is not None:
        completion_kwargs['custom_reasoning_level'] = reasoning_effort
    #
    if reasoning_effort is not None:
        completion_kwargs['custom_reasoning_level'] = reasoning_effort
    #
    try:
        # **completion_kwargs unpacks the dictionary into arguments
        if model_name in REASONING_MODELS:
            # Use the modified API call function for reasoning models
            response = client_to_use.responses.create(**completion_kwargs)
        else:
            # Use the standard chat completions API call
            response = client_to_use.chat.completions.create(**completion_kwargs)

        # capture response
        generated_text = response.choices[0].message.content
        # status message
        print("\nAPI call completed!")
        # content message
        print(f"Response from {model_name}: {generated_text}")

    except Exception as e:
        # error message
        print(f"\nAPI Call Failed for {model_name}. Error: {e}")

print("\n API-calls complete")

In [6]:
# @title Defintions for the API call

def _schema_to_tool(chain_of_thought: str) -> Tuple[str, Dict]:

    return {
        "type": "function",
        "function": {
            "name": "response_format",
            "description": "The function used to return the single, structured text response.",
            "parameters": {
                "type": "object",
                "properties": {
                    "final_response": {
                        "type": "string",
                        "description": (
                            f"Generate the complete, continuous peer-review based on the provided content. {chain_of_thought}"
                        )
                    }
                },
                "required": ["final_response"]
            }
        }
    }


# some chatgpt magic here - returns a dict parsed based on schema type
def _normalize_output(kind: str, parsed: dict, expect_item: Optional[str] = None) -> dict:
    def ci(x):
        try:
            return int(round(float(x)))
        except Exception:
            return None

    if not isinstance(parsed, dict):
        # Accept list of {k:v} pairs
        if isinstance(parsed, list):
            merged = {}
            for el in parsed:
                if isinstance(el, dict):
                    merged.update(el)
            parsed = merged
        else:
            parsed = {}

    if kind == "all_at_once":
        out = {k: ci(parsed.get(k)) for k in ITEMS}
        # clip for BJW
        for k in ["gut_x","gut_y"]:
            if out[k] is not None:
                out[k] = min(10, max(1, out[k]))
        for k in ["bjw1","bjw2","bjw3","bjw4","bjw5","bjw6"]:
            if out[k] is not None:
                out[k] = min(6, max(1, out[k]))
        return out

    if kind == "scale_gut":
        out = {k: ci(parsed.get(k)) for k in SCALES["gut"]}
        for k in out:
            if out[k] is not None:
                out[k] = min(10, max(1, out[k]))
        return out

    if kind == "scale_bjw":
        out = {k: ci(parsed.get(k)) for k in SCALES["bjw"]}
        for k in out:
            if out[k] is not None:
                out[k] = min(6, max(1, out[k]))
        return out

    # item
    q = parsed.get("question")
    v = parsed.get("response", parsed.get(q))  # tolerate {<item>: value}
    if q is None and expect_item:
        q = expect_item
    v = ci(v)
    if q in ["gut_x","gut_y"]:
        v = None if v is None else min(10, max(1, v))
    else:
        v = None if v is None else min(6, max(1, v))
    return {"question": q, "response": v}


In [ ]:
# @title API Handling
# adopted from Jamie

# function to parse JSON from a string and strip DeepSeek R1 reasoning if present
def _extract_json(text: str, model_name: str = "") -> dict:
    if not text:
        return {}

    # DeepSeek R1: strip 'thinking' section if present
    if "deepseek-r1-distill-llama-70b" in model_name:
        # common pattern: <think>...</think>{...} or reasoning text before JSON
        if "</think>" in text:
            text = text.split("</think>", 1)[-1]
        elif "<think>" in text:
            text = text.split("<think>", 1)[-1]
        # otherwise, try to cut at the first { or [ after any long text
        if "{" in text or "[" in text:
            idx = min(
                [i for i in [text.find("{"), text.find("[")] if i != -1]
            )
            text = text[idx:]

    # strip code fences if present
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    # find first {...} or [...]
    m = re.search(r"(\{.*\}|$begin:math:display$.*$end:math:display$)", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    try:
        return json.loads(text)
    except Exception:
        return {}

# fake clients for testing
class LLMClient:
    def __init__(self, simulate: bool = True, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)

        # placeholders for real clients when simulate=False:
        self._openai = None
        self._groq = None

    # condition on simulate=False
    def _maybe_init_clients(self):
        if self.simulate:
            return
        from openai import OpenAI
        self._openai = OpenAI(api_key=openai_api)
        from groq import Groq
        self._groq = Groq(api_key=groq_api)

    # define API call
    def call(
        self,
        model: str,
        prompt: str,
        temperature: Optional[float],
        reasoning_effort: Optional[ReasoningEffort],
        CoT: Optional[CoT],
        max_retries: int = 3,
        retry_delay: float = 1.5,
        ) -> Tuple[str, Dict]: # output raw text and the conditions

        self._maybe_init_clients()

        # random values for test sims
        if self.simulate:

          # construct input parameters
          payload = {
              "model": model,
              "temperature": temperature,
              "reasoning_effort": reasoning_effort,
              "CoT_dimension": CoT,
              "simulated_review": (
                  f"This is a simulated review: Model='{model}', Temp={temperature}, "
                  f"Effort='{reasoning_effort}', CoT='{CoT}'."
              )
          }

          # construct output parameter
          raw = json.dumps(payload, indent=2)

          # Immediately return the raw string and the structured dictionary
          return raw, payload

        provider = "groq" if model in GROQ_MODELS else "openai"

        for attempt in range(1, max_retries + 1):
            try:
                if provider == "openai":
                    # If an OpenAI "reasoning" model, use Responses API
                    # and rely on the contract + normalizer.
                    # Otherwise, use Chat Completions with tool-calling to enforce structure strictly.
                    if model in REASONING_MODELS:
                        kwargs = {}
                        if reasoning_effort:
                            kwargs["reasoning"] = {"effort": reasoning_effort}
                        if temperature is not None:
                            kwargs["temperature"] = float(temperature)

                        # instruction since we can't pass response_format:
                        contract = "Return exactly ONE JSON object" # altered from Jamie as I do not differ input of scales
                        resp = self._openai.responses.create(
                            model=model,
                            input=(prompt + contract),
                            **kwargs,
                        )
                        raw = getattr(resp, "output_text", None) or str(resp)

                    else:
                        # Chat Completions + tool calling (strict) for non-reasoning OpenAI models
                        tools = [_schema_to_tool(CoT)]

                        ckwargs = {}
                        if temperature is not None:
                            ckwargs["temperature"] = float(temperature)

                        # Add nudge to return only a tool call
                        system_msg = {
                            "role": "system",
                            "content": "You are a parser. Call the function with exactly one JSON object that includes a text."
                        }

                        msgs = [system_msg, {"role": "user", "content": prompt}]
                        resp = self._openai.chat.completions.create(
                            model=model,
                            messages=msgs,
                            tools=tools if tools else None,
                            **ckwargs,
                        )

                        choice = resp.choices[0]
                        # Prefer tool call arguments (strict JSON)
                        if choice.message.tool_calls:
                            args = choice.message.tool_calls[0].function.arguments
                            raw = args  # raw JSON text
                        else:
                            # Fallback to message content (should be JSON anyway)
                            raw = choice.message.content or ""

                else:
                    prompt2 = prompt + "Return exactly ONE JSON object" # altered from Jamie as I do not differ input of scales
                    gkwargs = {}
                    if temperature is not None and model not in REASONING_MODELS:
                        gkwargs["temperature"] = float(temperature)
                    resp = self._groq.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt2}],
                        **gkwargs,
                    )
                    raw = resp.choices[0].message.content.strip()

                # Parse + normalize
                parsed_raw = _extract_json(raw, model_name=model)
                normalized = _normalize_output(kind, parsed_raw, expect_item_single)
                return raw, normalized

            except Exception as e:
                print(f"[LLM ERROR] provider={provider} model={model} attempt={attempt} -> {e}", flush=True)
                if attempt == max_retries:
                    err_stub = f"ERROR: {type(e).__name__}: {e}"
                    return err_stub, {}

            time.sleep(retry_delay)



In [ ]:
# @title execution

# Dataset

In [ ]:
# @title Attach results